# 10-A - E10-A Reparacion local del codebook

E10-A prueba una reparacion **TTA local** del codebook: el checkpoint queda
congelado, pero el adapter mantiene una `codebook_view` reparable durante
inferencia. El objetivo es diagnosticar si la memoria-codebook puede volverse
util bajo shift sin volver a entrenar ni guardar un nuevo checkpoint.


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'dememte').exists():
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
print('repo root:', ROOT)


repo root: /home/nakato/projects/Dememte


In [2]:
import json
import numpy as np
import pandas as pd
import torch

from dememte.codebook_repair import (
    CodebookRepairConfig,
    LocalCodebookRepairAdapter,
    calibrate_repair_thresholds,
)
from dememte.config import e6_config, resolve_data_dir
from dememte.data import build_loaders, seed_everything
from dememte.evaluation import evaluate_dememte_suite, evaluate_dememte_tta_suite, signal_curve_rows
from dememte.io import ensure_dir, load_checkpoint, write_csv, write_json
from dememte.models import make_dememte_e6
from dememte.retrieval import RetrievalCache

BASE = 'e6_ema_kmeans_restart'
OUT = ensure_dir(ROOT / 'notebooks' / '10a_codebook_repair' / 'out')
E6_OUT = ROOT / 'notebooks' / '06_e6_zq_alignment' / 'out'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)


device: cuda


## Data and checkpoint


In [3]:
cfg = e6_config(BASE)
cfg.data_dir = resolve_data_dir(cfg)
seed_everything(cfg.seed)

tr_loader, va_loader, te_loader, meta = build_loaders(
    data_dir=cfg.data_dir,
    batch_size=cfg.batch_size,
    num_workers=cfg.num_workers,
    val_ratio=cfg.val_ratio,
    split_seed=cfg.split_seed,
    protocol=cfg.benchmark_protocol,
)
print(meta)

ckpt = E6_OUT / BASE / 'best.pt'

def load_base_model():
    model = make_dememte_e6(cfg, device=device)
    load_checkpoint(model, ckpt, device=device, strict=False)
    model.eval()
    return model


{'protocol': 'historical_trainval_resplit', 'split_seed': 42, 'train_size': 1632, 'val_size': 408, 'test_size': 6149}


## Phase 0 - clean/source reliability thresholds


In [4]:
phase0_model = load_base_model()
thresholds = calibrate_repair_thresholds(
    phase0_model,
    tr_loader,
    device=device,
    max_batches=8,
    margin_quantile=0.25,
    dq_quantile=0.75,
)
write_json(thresholds, OUT / 'e10a_phase0_thresholds.json')
thresholds


{'margin_min': 1.0837111473083496, 'dq_max': 0.3122282326221466}

## Variant matrix


In [5]:
base_kwargs = dict(
    usage_decay=0.99,
    dead_threshold=1.0,
    repair_every=25,
    max_reseeds=16,
    lambda_max=0.5,
    margin_min=thresholds['margin_min'],
    dq_max=thresholds['dq_max'],
)
variants = [
    ('repair_replace_ema',
     CodebookRepairConfig(repair_mode='ema', blend_mode='replace', ema_lr=0.05,
                          anchor_strength=0.0, max_code_drift=0.5, **base_kwargs)),
    ('repair_gated_ema',
     CodebookRepairConfig(repair_mode='ema', blend_mode='gated', ema_lr=0.05,
                          anchor_strength=0.0, max_code_drift=0.5, **base_kwargs)),
    ('repair_reseed_only',
     CodebookRepairConfig(repair_mode='reseed', blend_mode='gated', ema_lr=0.0,
                          anchor_strength=0.0, max_code_drift=0.5, **base_kwargs)),
    ('repair_ema_reseed',
     CodebookRepairConfig(repair_mode='ema_reseed', blend_mode='gated', ema_lr=0.05,
                          anchor_strength=0.0, max_code_drift=0.5, **base_kwargs)),
    ('repair_ema_reseed_anchored',
     CodebookRepairConfig(repair_mode='ema_reseed', blend_mode='gated', ema_lr=0.05,
                          anchor_strength=0.01, max_code_drift=0.5, **base_kwargs)),
    ('repair_disabled_identity_check',
     CodebookRepairConfig(repair_mode='disabled', blend_mode='source', ema_lr=0.0,
                          anchor_strength=0.0, max_code_drift=0.0, **base_kwargs)),
]
[(name, v.repair_mode, v.blend_mode) for name, v in variants]


[('repair_replace_ema', 'ema', 'replace'),
 ('repair_gated_ema', 'ema', 'gated'),
 ('repair_reseed_only', 'reseed', 'gated'),
 ('repair_ema_reseed', 'ema_reseed', 'gated'),
 ('repair_ema_reseed_anchored', 'ema_reseed', 'gated'),
 ('repair_disabled_identity_check', 'disabled', 'source')]

## Run E10-A


In [6]:
def write_markdown_table(rows, path):
    if not rows:
        path.write_text('', encoding='utf-8')
        return
    df = pd.DataFrame(rows)
    cols = [
        'variant', 'clean_acc', 'corrupt_acc_avg', 'delta_corrupt_vs_source',
        'ece_corrupt_avg', 'nll_corrupt_avg', 'repair_hard_usage_corrupt_avg',
        'repair_dead_code_fraction_corrupt_avg', 'repair_codebook_drift_corrupt_avg',
        'repair_dq_delta_corrupt_avg', 'repair_reliability_corrupt_avg',
    ]
    cols = [c for c in cols if c in df.columns]
    path.write_text(df[cols].to_markdown(index=False), encoding='utf-8')

all_summaries = []
all_curves = []

src_model = load_base_model()
src_metrics = evaluate_dememte_suite(src_model, te_loader, device=device)
src_clean = src_metrics.pop('clean_record')
src_corrupt = src_metrics.pop('corruption_records')
src_summary = {k: v for k, v in src_metrics.items() if isinstance(v, (int, float, bool, str, np.floating))}
src_summary.update({
    'variant': 'source',
    'label': f'{BASE}::source',
    'base_variant': BASE,
    'base_checkpoint': str(ckpt),
    'protocol': meta['protocol'],
    'split_seed': meta['split_seed'],
    'quantizer_type': cfg.quantizer_type,
    'delta_clean_vs_source': 0.0,
    'delta_corrupt_vs_source': 0.0,
})
all_summaries.append(src_summary)
all_curves.extend(signal_curve_rows('source', src_summary['label'], src_clean, src_corrupt))

for variant_name, repair_cfg in variants:
    print('--', variant_name)

    def factory(repair_cfg=repair_cfg):
        return LocalCodebookRepairAdapter(load_base_model(), repair_cfg)

    metrics = evaluate_dememte_tta_suite(
        factory,
        te_loader,
        device=device,
        tta_method=variant_name,
        tta_base_variant=BASE,
    )
    clean_record = metrics.pop('clean_record')
    corrupt_records = metrics.pop('corruption_records')
    label = f'{BASE}::{variant_name}'
    curve_rows = signal_curve_rows(variant_name, label, clean_record, corrupt_records)
    summary = {k: v for k, v in metrics.items() if isinstance(v, (int, float, bool, str, np.floating))}
    summary.update({
        'variant': variant_name,
        'label': label,
        'base_variant': BASE,
        'base_checkpoint': str(ckpt),
        'protocol': meta['protocol'],
        'split_seed': meta['split_seed'],
        'quantizer_type': cfg.quantizer_type,
        'delta_clean_vs_source': metrics['clean_acc'] - src_summary['clean_acc'],
        'delta_corrupt_vs_source': metrics['corrupt_acc_avg'] - src_summary['corrupt_acc_avg'],
    })
    all_summaries.append(summary)
    all_curves.extend(curve_rows)

    method_dir = ensure_dir(OUT / variant_name)
    write_json(summary, method_dir / 'metrics.json')
    write_csv(curve_rows, method_dir / 'signal_curves.csv')
    print({
        k: round(float(summary[k]), 4)
        for k in ['clean_acc', 'corrupt_acc_avg', 'delta_corrupt_vs_source',
                  'repair_hard_usage_corrupt_avg', 'repair_dq_delta_corrupt_avg']
        if k in summary
    })

ranked = sorted(all_summaries, key=lambda r: r.get('corrupt_acc_avg', 0.0), reverse=True)
write_csv(ranked, OUT / 'e10a_results.csv')
write_csv(all_curves, OUT / 'e10a_curves.csv')
write_markdown_table(ranked, OUT / 'e10a_summary.md')
pd.DataFrame(ranked).head(20)


-- repair_replace_ema
{'clean_acc': 0.76, 'corrupt_acc_avg': 0.509, 'delta_corrupt_vs_source': 0.006, 'repair_hard_usage_corrupt_avg': 0.0141, 'repair_dq_delta_corrupt_avg': -0.0092}
-- repair_gated_ema
{'clean_acc': 0.7544, 'corrupt_acc_avg': 0.5046, 'delta_corrupt_vs_source': 0.0016, 'repair_hard_usage_corrupt_avg': 0.0141, 'repair_dq_delta_corrupt_avg': -0.0092}
-- repair_reseed_only
{'clean_acc': 0.7525, 'corrupt_acc_avg': 0.503, 'delta_corrupt_vs_source': 0.0, 'repair_hard_usage_corrupt_avg': 0.0145, 'repair_dq_delta_corrupt_avg': -0.0}
-- repair_ema_reseed
{'clean_acc': 0.7544, 'corrupt_acc_avg': 0.5045, 'delta_corrupt_vs_source': 0.0015, 'repair_hard_usage_corrupt_avg': 0.014, 'repair_dq_delta_corrupt_avg': -0.0092}
-- repair_ema_reseed_anchored
{'clean_acc': 0.7546, 'corrupt_acc_avg': 0.5048, 'delta_corrupt_vs_source': 0.0018, 'repair_hard_usage_corrupt_avg': 0.014, 'repair_dq_delta_corrupt_avg': -0.0092}
-- repair_disabled_identity_check
{'clean_acc': 0.7523, 'corrupt_acc_avg'

,clean_acc,corrupt_acc_avg,corrupt_acc_gaussian_noise,corrupt_acc_pixel_mask,corrupt_acc_cutout,corrupt_acc_blur,ece_clean,ece_corrupt_avg,nll_clean,nll_corrupt_avg,...,repair_reliability_corrupt_avg,variant,label,base_variant,base_checkpoint,protocol,split_seed,quantizer_type,delta_clean_vs_source,delta_corrupt_vs_source
0,0.759961,0.508972,0.359679,0.353716,0.645633,0.676858,0.054341,0.086083,0.944309,1.986352,...,0.497124,repair_replace_ema,e6_ema_kmeans_restart::repair_replace_ema,e6_ema_kmeans_restart,/home/nakato/projects/Dememte/notebooks/06_e6_...,historical_trainval_resplit,42,ema_vq,0.007644,0.006017
1,0.754594,0.504798,0.354638,0.350355,0.641188,0.673009,0.056837,0.088965,0.967478,2.012157,...,0.497639,repair_ema_reseed_anchored,e6_ema_kmeans_restart::repair_ema_reseed_anchored,e6_ema_kmeans_restart,/home/nakato/projects/Dememte/notebooks/06_e6_...,historical_trainval_resplit,42,ema_vq,0.002277,0.001843
2,0.754432,0.504554,0.354150,0.350301,0.641080,0.672684,0.056979,0.089209,0.967699,2.012454,...,0.497124,repair_gated_ema,e6_ema_kmeans_restart::repair_gated_ema,e6_ema_kmeans_restart,/home/nakato/projects/Dememte/notebooks/06_e6_...,historical_trainval_resplit,42,ema_vq,0.002114,0.001599
3,0.754432,0.504486,0.354204,0.350301,0.640971,0.672467,0.056950,0.089276,0.967775,2.012453,...,0.497148,repair_ema_reseed,e6_ema_kmeans_restart::repair_ema_reseed,e6_ema_kmeans_restart,/home/nakato/projects/Dememte/notebooks/06_e6_...,historical_trainval_resplit,42,ema_vq,0.002114,0.001531
4,0.752480,0.502982,0.353553,0.348783,0.638749,0.670841,0.058084,0.090277,0.976958,2.022327,...,0.481570,repair_reseed_only,e6_ema_kmeans_restart::repair_reseed_only,e6_ema_kmeans_restart,/home/nakato/projects/Dememte/notebooks/06_e6_...,historical_trainval_resplit,42,ema_vq,0.000163,0.000027
5,0.752317,0.502954,0.353445,0.348675,0.638911,0.670787,0.058221,0.090268,0.976969,2.022439,...,NaN,source,e6_ema_kmeans_restart::source,e6_ema_kmeans_restart,/home/nakato/projects/Dememte/notebooks/06_e6_...,historical_trainval_resplit,42,ema_vq,0.000000,0.000000
6,0.752317,0.502954,0.353445,0.348675,0.638911,0.670787,0.058221,0.090268,0.976969,2.022439,...,0.000000,repair_disabled_identity_check,e6_ema_kmeans_restart::repair_disabled_identit...,e6_ema_kmeans_restart,/home/nakato/projects/Dememte/notebooks/06_e6_...,historical_trainval_resplit,42,ema_vq,0.000000,0.000000


## Probe - zq source vs zq repaired retrieval utility


In [7]:
@torch.no_grad()
def collect_zq_keys(forwarder, loader, key='zq_pool'):
    keys, labels = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        if isinstance(forwarder, LocalCodebookRepairAdapter):
            _, dbg = forwarder(x, return_debug=True)
        else:
            _, _, dbg = forwarder(x, return_debug=True)
        keys.append(dbg[key].detach().cpu())
        labels.append(y.detach().cpu().long())
    return torch.cat(keys, dim=0), torch.cat(labels, dim=0)

@torch.no_grad()
def retrieval_probe(cache_keys, cache_labels, query_keys, query_labels, num_classes=102, top_k=16, beta=5.0):
    cache = RetrievalCache(cache_keys, cache_labels, num_classes=num_classes)
    info = cache.query(query_keys, top_k=top_k, beta=beta)
    pred = info['logits'].argmax(dim=1).cpu()
    return {
        'probe_acc': float((pred == query_labels).float().mean().item()),
        'probe_margin_mean': float(info['margin'].mean().item()),
        'probe_entropy_mean': float(info['entropy'].mean().item()),
    }

probe_cfg = CodebookRepairConfig(
    repair_mode='ema_reseed',
    blend_mode='gated',
    ema_lr=0.05,
    anchor_strength=0.01,
    max_code_drift=0.5,
    **base_kwargs,
)
source_train_keys, source_train_y = collect_zq_keys(load_base_model(), tr_loader, key='zq_pool')
source_test_keys, source_test_y = collect_zq_keys(load_base_model(), te_loader, key='zq_pool')
repaired_train_adapter = LocalCodebookRepairAdapter(load_base_model(), probe_cfg)
repaired_test_adapter = LocalCodebookRepairAdapter(load_base_model(), probe_cfg)
repaired_train_keys, repaired_train_y = collect_zq_keys(repaired_train_adapter, tr_loader, key='zq_pool_repaired')
repaired_test_keys, repaired_test_y = collect_zq_keys(repaired_test_adapter, te_loader, key='zq_pool_repaired')

probe = {
    'source_zq': retrieval_probe(source_train_keys, source_train_y, source_test_keys, source_test_y, cfg.num_classes),
    'repaired_zq': retrieval_probe(repaired_train_keys, repaired_train_y, repaired_test_keys, repaired_test_y, cfg.num_classes),
}
probe['delta_probe_acc'] = probe['repaired_zq']['probe_acc'] - probe['source_zq']['probe_acc']
write_json(probe, OUT / 'e10a_retrieval_probe.json')
probe


{'source_zq': {'probe_acc': 0.5529354214668274,
  'probe_margin_mean': 3.4765803813934326,
  'probe_entropy_mean': 1.5227739810943604},
 'repaired_zq': {'probe_acc': 0.5740770697593689,
  'probe_margin_mean': 3.6336076259613037,
  'probe_entropy_mean': 1.4805223941802979},
 'delta_probe_acc': 0.021141648292541504}

## Interpretation guardrails

E10-A is successful only if the repaired codebook improves internal
codebook health **and** makes `zq_pool` less harmful as memory. Accuracy
alone is not enough; the probe and `repair_*` diagnostics carry the main
claim.
